# COMP 551 Assignment 3

## Task 1: Acquire the data

In [1]:
#!pip install torchvision

In [2]:
import torch
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader

### **Step 1: Load Dataset**

In [3]:
transform=transforms.ToTensor()
train_dataset_raw = datasets.FashionMNIST(root='./data', train=True, download=True,
                                      transform=transforms.ToTensor())
test_dataset_raw = datasets.FashionMNIST(root='./data', train=False, download=True)

100%|██████████| 26.4M/26.4M [00:02<00:00, 10.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 246kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.63MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 2.48MB/s]


In [4]:
image, label = train_dataset_raw[0]
print(type(image))
print(image.shape)
print(image.dtype)
print(image.device)
print(f"Train partition: {len(train_dataset_raw)} images")
print(f"Test partition:  {len(test_dataset_raw)} images")

<class 'torch.Tensor'>
torch.Size([1, 28, 28])
torch.float32
cpu
Train partition: 60000 images
Test partition:  10000 images


### **Step 2: Normalize Dataset**

In [5]:
sum = 0.0
squared_sum = 0.0
N = 0
train_loader = DataLoader(train_dataset_raw, batch_size=1000, shuffle=False)
for data, _ in train_loader: # Accumulative sum and std throughout data
    sum += data.sum(dim=[0, 2, 3])
    squared_sum += (data ** 2).sum(dim=[0, 2, 3])
    N += data.numel()
mean = (sum / N).item()
std = (torch.sqrt((squared_sum / N) - (mean ** 2))).item()
print(mean)
print(std)

0.28604063391685486
0.3530242443084717


In [6]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((mean,), (std,))
])
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True,
                                      transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True,
                                      transform=transform)                                     

### **Step 3: Create Training and Validation Set**

In [7]:
from torch.utils.data import random_split

split_ratio = 0.8
train_size = int(split_ratio * len(train_dataset))
val_size = len(train_dataset) - train_size

train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

print(f"Train size: {train_size}")
print(f"Validation size: {val_size}")

Train size: 48000
Validation size: 12000


In [8]:
from torch.utils.data import DataLoader

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

## Task 2: Implement a Multilayer Perceptron